# Init

In [0]:
import sys
sys.path.append("../..")

from common.io import write_table
from common.config import gold_table

# Read from Silver Table/s

In [0]:
%sql
SELECT
    ROW_NUMBER() OVER(ORDER BY cc.customer_id) AS customer_key,
    cc.customer_id,
    cc.customer_key,
    cc.firstname,
    cc.lastname,
    el.country,
    cc.marital_status,
    CASE
        WHEN cc.gender <> 'N/A' THEN cc.gender
        ELSE COALESCE(ec.gender, 'N/A')
    END AS gender,
    ec.birth_date,
    cc.create_date AS created_at
FROM baraa_dev_project.silver.crm_customers cc
LEFT JOIN baraa_dev_project.silver.erp_customers ec ON cc.customer_key = ec.customer_id
LEFT JOIN baraa_dev_project.silver.erp_locations el ON cc.customer_key = el.customer_id

# Business transformation and modeling

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER(ORDER BY cc.customer_id) AS customer_key,
    cc.customer_id,
    cc.customer_key AS customer_number,
    cc.firstname,
    cc.lastname,
    el.country,
    cc.marital_status,
    CASE
        WHEN cc.gender <> 'N/A' THEN cc.gender
        ELSE COALESCE(ec.gender, 'N/A')
    END AS gender,
    ec.birth_date,
    cc.create_date AS created_at
FROM baraa_dev_project.silver.crm_customers cc
LEFT JOIN baraa_dev_project.silver.erp_customers ec ON cc.customer_key = ec.customer_id
LEFT JOIN baraa_dev_project.silver.erp_locations el ON cc.customer_key = el.customer_id
"""
df = spark.sql(query)

# Write it to Gold Table

In [0]:
write_table(df, gold_table('dim_customers'))

In [0]:
%sql
SELECT * FROM baraa_dev_project.gold.dim_customers